# LEGA-C DR3: galaxies lost for reasons other than being passive

- Passive and quality cuts as in `results/legac-dr3-2026-11-11/analysis.ipynb`.
- Photometry flags: COSMOS2015 clean cut against the fit rule of `scripts/cosmos_photometry.py`.

In [1]:
import json, warnings
from pathlib import Path
import astropy.units as u
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from scipy.stats import ks_2samp
from specutils.utils.wcs_utils import air_to_vac

warnings.filterwarnings('ignore')
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
RAW, OUT = ROOT / 'data/raw', ROOT / 'results/dr3-more-galaxies-2026-11-16'
WINDOW, TELLURIC = (4074.5, 4666.1), (7590.0, 7660.0)
text = lambda v: (v.decode() if isinstance(v, bytes) else str(v)).strip()

cat = Table.read(RAW / 'legac_dr3/legac_DR3.fits', hdu=1).to_pandas()
for column in ('SPECT_ID', 'Filename'):
    cat[column] = cat[column].map(text)
phot = Table.read(RAW / 'cosmos2015/cosmos2015_legac_dr3_photometry_1arcsec.fits').to_pandas().set_index('LEGAC_INDEX')
P = cat.join(phot[['Area', 'Sat', 'Cfl', 'Deep', 'Flag', 'NUVMag', 'RMag', 'JMag']], how='inner')

def flag(path, column):
    t = Table.read(path)
    return dict(zip(np.char.strip(np.asarray(t['SPECT_ID']).astype(str)), np.asarray(t[column]).astype(int)))

P['warn25'] = P.SPECT_ID.map(flag(RAW / 'cosmos2025/cosmos2025_phot_legac_dr3_1arcsec.fits', 'warn-flag'))
P['classic'] = P.SPECT_ID.map(flag(RAW / 'cosmos2020/cosmos2020_classic_legac_dr3_1arcsec.fits', 'FlagCOMBINED'))
# Fit rule: COSMOS2025 where matched (warn-flag 0), otherwise Classic (FlagCOMBINED 0).
P['fittable'] = np.where(P.warn25.notna(), P.warn25 == 0, P.classic == 0)
P['photometry'] = np.where(P.warn25.notna(), 'cosmos2025', 'cosmos2020_classic')
P['c15_clean'] = (P.Area == 0) & (P.Sat == 0) & (P.Cfl == 1) & (P.Flag == 0)

- Quality, NUVrJ, [O II] EW and 3σ line cuts, unchanged.

In [2]:
quality = (P.FLAG_PRIMARY == 1) & (P.FLAG_SPEC == 0) & P.Z.notna() & P.SIGMA_STARS_PRIME.notna() & (P.FLAG_MORPH == 0)
quality &= (P.SN > 0) & (P.Z >= 0.6) & (P.Z < 1.0) & (P[['NUVMag', 'RMag', 'JMag']] > -40).all(axis=1)
P['nuv_r'], P['r_j'] = P.NUVMag - P.RMag, P.RMag - P.JMag
nuvrj = (P.nuv_r > 3 * P.r_j + 1) & (P.nuv_r > 3.1)
oii_ew = (P.OII_3727_EW < 5) | P.OII_3727_EW.isna()
detected = lambda line: (P[f'{line}_EW'] / P[f'{line}_EW_ERR']).abs() >= 3
no_line = ~(detected('OII_3727') | detected('OIII_5007'))
S = P[quality & nuvrj & oii_ew & no_line].copy()
chain = [('COSMOS2015 match', len(P)), ('quality', quality.sum()), ('NUVrJ quiescent', (quality & nuvrj).sum()),
         ('[O II] EW < 5 A', (quality & nuvrj & oii_ew).sum()), ('no 3 sigma [O II] or [O III]', len(S))]

- Good-pixel rest-frame span outside the telluric band; window coverage.

In [3]:
def rest_span(filename, z):
    with fits.open(RAW / 'legac_dr3/sp' / filename) as hdulist:
        wave, flux, err, qual = (np.asarray(hdulist[1].data[c][0], float) for c in ('WAVE', 'FLUX', 'ERR', 'QUAL'))
    good = (qual == 0) & (err > 0) & np.isfinite(flux) & ~((wave >= TELLURIC[0]) & (wave <= TELLURIC[1]))
    if not good.any():
        return np.nan, np.nan
    rest = air_to_vac(wave[good] * u.AA).to_value(u.AA) / (1 + z)
    return rest.min(), rest.max()

S[['rest_min', 'rest_max']] = [rest_span(f, z) for f, z in zip(S.Filename, S.Z)]
S['covers'] = (S.rest_min <= WINDOW[0]) & (S.rest_max >= WINDOW[1])
S['sigma_err_finite'] = np.isfinite(S.SIGMA_STARS_PRIME_ERR)
one_per_object = lambda d: d.sort_values(['SN', 'SPECT_ID'], ascending=[False, True]).drop_duplicates('OBJECT', keep='first')

- Reference chain: reproduces the 194.
- New chain: fittable photometry replaces the COSMOS2015 cut; one spectrum per OBJECT picked last.

In [4]:
old = one_per_object(S[S.c15_clean])
old = old[old.fittable & old.covers & old.sigma_err_finite]
reference = json.loads((ROOT / 'results/legac-dr3-2026-11-11/window-194/experiment.json').read_text())['targets']
assert sorted(old.SPECT_ID) == sorted(reference)

new = one_per_object(S[S.fittable & S.covers])
gained = new[~new.OBJECT.isin(old.OBJECT)].copy()
gained['change'] = np.select(
    [~gained.c15_clean & gained.sigma_err_finite, gained.c15_clean & ~gained.sigma_err_finite, ~gained.c15_clean],
    ['COSMOS2015 flag', 'NaN sigma-star error', 'COSMOS2015 flag + NaN sigma-star error'], 'spectrum picked after window cut')
chain += [('fittable photometry', S.fittable.sum()), ('covers window', (S.fittable & S.covers).sum()),
          ('one per OBJECT', len(new)), ('of which the reference 194', len(old)), ('gained', len(gained))]
chain += [(f'gained: {name}', n) for name, n in gained.change.value_counts().items()]
pd.DataFrame(chain, columns=['step', 'n']).to_csv(OUT / 'selection-counts.csv', index=False)
pd.DataFrame(chain, columns=['step', 'n'])

,step,n
0,COSMOS2015 match,4057
1,quality,2571
2,NUVrJ quiescent,1222
3,[O II] EW < 5 A,931
4,no 3 sigma [O II] or [O III],581
5,fittable photometry,523
6,covers window,371
7,one per OBJECT,348
8,of which the reference 194,194
9,gained,154


- COSMOS2015 flags of the 252 spectra the clean cut removes (VizieR J/ApJS/224/24: `Area` FLAG_HJMCC, `Sat` FLAG_PETER, `Cfl` FLAG_COSMOS, `Flag` FLAG_SHALLOW).

In [5]:
removed = S[~S.c15_clean]
flags = removed.groupby(['Area', 'Sat', 'Cfl', 'Flag']).agg(spectra=('fittable', 'size'), fittable=('fittable', 'sum')).reset_index()
flags.to_csv(OUT / 'cosmos2015-flags-removed.csv', index=False)
flags

,Area,Sat,Cfl,Flag,spectra,fittable
0,0,0,0,0,3,3
1,0,0,1,1,208,194
2,0,1,1,0,3,1
3,0,1,1,1,3,3
4,0,2,1,0,3,2
5,0,2,1,1,1,1
6,0,3,1,0,3,0
7,0,3,1,1,1,0
8,0,4,1,0,2,1
9,0,5,1,0,1,0


- σ★ error stand-in: median fractional error of the 194.

In [6]:
FRACTION = float((old.SIGMA_STARS_PRIME_ERR / old.SIGMA_STARS_PRIME).median())
FRACTION

0.03883352875709534

- Gained against the 194: 16, 50, 84 % and two-sample KS p.

In [7]:
for d in (old, gained):
    d['nuvrj_margin'] = np.minimum(d.nuv_r - (3 * d.r_j + 1), d.nuv_r - 3.1)
    d['oii_snr'] = (d.OII_3727_EW / d.OII_3727_EW_ERR).abs()
    d['oiii_snr'] = (d.OIII_5007_EW / d.OIII_5007_EW_ERR).abs()
COLUMNS = ['Z', 'SN', 'nuv_r', 'r_j', 'nuvrj_margin', 'OII_3727_EW', 'oii_snr', 'OIII_5007_EW', 'oiii_snr',
           'LICK_HD_A', 'SIGMA_STARS_PRIME']
rows = []
for column in COLUMNS:
    a, b = old[column].dropna(), gained[column].dropna()
    rows.append(dict(quantity=column, n_194=len(a), **dict(zip(['q16_194', 'q50_194', 'q84_194'], np.quantile(a, [.16, .5, .84]))),
                     n_gained=len(b), **dict(zip(['q16_gained', 'q50_gained', 'q84_gained'], np.quantile(b, [.16, .5, .84]))),
                     ks_p=ks_2samp(a, b).pvalue))
compare = pd.DataFrame(rows)
compare.to_csv(OUT / 'gained-vs-194.csv', index=False)
compare.round(3)

,quantity,n_194,q16_194,q50_194,q84_194,n_gained,q16_gained,q50_gained,q84_gained,ks_p
0,Z,194,0.669,0.728,0.842,154,0.665,0.728,0.836,0.128
1,SN,194,11.200,20.700,31.344,154,12.000,21.400,34.352,0.699
2,nuv_r,194,4.438,5.010,5.362,154,4.450,4.920,5.380,0.451
3,r_j,194,0.810,0.900,1.020,154,0.815,0.915,1.020,0.816
4,nuvrj_margin,194,0.638,1.280,1.721,154,0.610,1.185,1.706,0.282
5,OII_3727_EW,109,0.125,0.682,1.912,86,-0.069,0.331,1.052,0.025
6,oii_snr,109,0.374,1.336,2.502,86,0.353,0.870,2.161,0.037
7,OIII_5007_EW,117,0.070,0.338,0.728,92,0.089,0.273,0.618,0.235
8,oiii_snr,117,0.329,1.305,2.288,92,0.381,1.177,2.037,0.761
9,LICK_HD_A,169,-0.762,0.331,1.878,144,-0.832,0.319,1.594,0.961


- Target list for the fits.

In [8]:
KEEP = ['SPECT_ID', 'OBJECT', 'Z', 'SN', 'change', 'photometry', 'Area', 'Sat', 'Cfl', 'Deep', 'Flag', 'nuv_r', 'r_j',
        'OII_3727_EW', 'OII_3727_EW_ERR', 'OIII_5007_EW', 'OIII_5007_EW_ERR', 'SIGMA_STARS_PRIME', 'SIGMA_STARS_PRIME_ERR',
        'rest_min', 'rest_max']
gained = gained.sort_values('SPECT_ID')
gained[KEEP].to_csv(OUT / 'gained-per-galaxy.csv', index=False)
(OUT / 'targets.json').write_text(json.dumps({'targets': [
    {'spect_id': r.SPECT_ID, 'object_id': int(r.OBJECT), 'change': r.change} for r in gained.itertuples()]}, indent=1) + '\n')
gained.change.value_counts()

change
COSMOS2015 flag                           134
NaN sigma-star error                       14
COSMOS2015 flag + NaN sigma-star error      4
spectrum picked after window cut            2
Name: count, dtype: int64

## Fits of the gained galaxies
- Arms `dr3_cn_free_w` and `dr3_plain_w` of `results/legac-dr3-2026-11-11/window-194/experiment.json`, seed 20260832.
- $\sigma_\star$ error = `FRACTION` $\times\,\sigma_\star$ where the catalogue error is NaN (`sigma_err_fraction`).

In [9]:
import re, sys
import h5py
import jax
from astropy.cosmology import Planck18
from scipy.stats import linregress
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'scripts'))
from scripts.experiment import validate_result
from absorption_mask_analysis import weighted_quantiles
from ceridwen.cosmology import age_gyr, resolve_cosmology
from ceridwen.fit import load_result_h5
from src.chronometer import effective_redshift

jax.config.update('jax_enable_x64', True)
REFERENCE = ROOT / 'results/legac-dr3-2026-11-11/window-194/run/fits'
# arm: fit folder name, run folders of the gained galaxies
ARMS = {'cn_free': ('dr3_cn_free_w', ('first-10', 'rest-a', 'rest-b')), 'plain': ('dr3_plain_w', ('plain-colab', 'plain-modal'))}
F_FIXED = 0.03  # one calibration floor for every fit, so chi2 compares
UNIFORM = re.compile(r'^\s*(afe|cfe|nfe|logzsol|log_f_calib): shape \(1,\); prior Uniform\(([-\d.]+), ([-\d.]+)\)', re.M)
# Pull = (observed - model) / error per band. UltraVISTA: the bands the COSMOS2015 shallow flag marks; broad optical: the saturation masks.
BAND_GROUPS = {'ultravista': 'vista_vircam', 'broad_optical': r'hsc_|megacam|suprimecam_(?:B|V|rp|ip|zp)$', 'irac': 'irac'}
SUMMARY = {'age_gyr': 'mass-weighted age [Gyr]', 'afe': 'afe', 'feh': 'logzsol', 'floor_pct': 'calibration floor [%]',
           'sigma_smooth': 'sigma_smooth', 'log_f_calib': 'log_f_calib'}


def fit_row(directory):
    row = dict(path=str(directory.relative_to(ROOT)))
    try:
        validate_result(directory)
    except Exception as error:
        return row | dict(valid=False, failure=str(error).split(':')[0])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        a, d = f.attrs, f['diagnostics'].attrs
        row |= dict(valid=True, failure='', z=float(a['redshift']), seed=int(a['random_seed']), sn=float(a['catalogue_sn']),
                    photometry=a['photometry_source'], sigma_star=float(a['sigma_star_kms']), sigma_star_err=float(a['sigma_star_err']),
                    phot_chi2=float(d['photometry_chi2']), phot_n=int(d['photometry_ndof']), spec_n=int(d['spectrum_ndof']))
        s = {k: f[f'spectrum/{k}'][:] for k in ('observed', 'uncertainty', 'posterior_q50')}
        m = f['spectrum/mask'][:].astype(bool)
        bands = pd.Series(f['photometry/pull'][:], index=f['photometry/filters'].asstr()[:])[f['photometry/mask'][:]]
        bounds = {n: (float(lo), float(hi)) for n, lo, hi in UNIFORM.findall(a['parameter_block'])}
    resid = (s['observed'] - s['posterior_q50'])[m]
    row |= dict(chi2px_raw=float(np.mean((resid / s['uncertainty'][m]) ** 2)),
                chi2px_fixed=float(np.mean((resid / np.hypot(s['uncertainty'], F_FIXED * np.abs(s['posterior_q50']))[m]) ** 2)),
                phot_chi2_per_band=row['phot_chi2'] / row['phot_n'], phot_pull_mean=bands.mean(), phot_pull_max=bands.abs().max(),
                **{f'{label}_pull_mean': bands[bands.index.str.contains(pattern)].mean() for label, pattern in BAND_GROUPS.items()})
    for label, name in SUMMARY.items():
        row |= {f'{label}_{k}': float(q[k][name]) for k in q}
    with h5py.File(directory / 'ceridwen_result.h5') as f:
        row |= dict(lnz=float(f['samples'].attrs['log_evidence']), lnz_err=float(f['samples'].attrs['log_evidence_err']))
    fit = load_result_h5(directory / 'ceridwen_result.h5')
    for p in ('cfe', 'nfe') if 'cfe' in fit.samples else ():
        row |= dict(zip((f'{p}_q16', f'{p}_q50', f'{p}_q84'), weighted_quantiles(np.asarray(fit.samples[p]), np.asarray(fit.log_weights))))
    row['sigma_resid'] = (row['sigma_smooth_q50'] - row['sigma_star']) / row['sigma_star_err']
    # Railed: posterior median within 5 % of the prior range from a bound; sigma: beyond 2.5 of the 3 sigma clip.
    for p, (lo, hi) in bounds.items():
        col = {'logzsol': 'feh'}.get(p, p)
        row[f'{col}_railed'] = bool(min(row[f'{col}_q50'] - lo, hi - row[f'{col}_q50']) < 0.05 * (hi - lo))
    row['sigma_railed'] = bool(abs(row['sigma_resid']) > 2.5)
    return row


def gained_dir(t, arm_dir, runs):
    name = f"{t['object_id']}-{t['spect_id']}"
    return next(d for run in runs if (d := OUT / run / 'run/fits' / arm_dir / name).exists())


W194 = json.loads((ROOT / 'results/legac-dr3-2026-11-11/targets.json').read_text())['targets']
GAINED = json.loads((OUT / 'targets.json').read_text())['targets']
rows = []
for arm, (arm_dir, runs) in ARMS.items():
    rows += [dict(arm=arm, sample='194', change='', target=t['spect_id'], object_id=t['object_id'])
             | fit_row(REFERENCE / arm_dir / f"{t['object_id']}-{t['spect_id']}") for t in W194]
    rows += [dict(arm=arm, sample='gained', change=t['change'], target=t['spect_id'], object_id=t['object_id'])
             | fit_row(gained_dir(t, arm_dir, runs)) for t in GAINED]
fit_table = pd.DataFrame(rows)
flags = pd.read_csv(OUT / 'gained-per-galaxy.csv').set_index('SPECT_ID')
masked = (flags.Sat > 0) | (flags.Area != 0) | (flags.Cfl != 1)
cosmos2015 = pd.Series(np.select([masked, flags.Flag == 1], ['saturation or bad area', 'shallow only'], 'clean'), index=flags.index)
fit_table.insert(3, 'cosmos2015', fit_table.target.map(cosmos2015).fillna('clean'))
fit_table.to_csv(OUT / 'fits.csv', index=False)
ok = fit_table[fit_table.valid].copy()
assert (ok.seed == 20260832).all()
nan_err = ok[ok.change.str.contains('NaN')]
assert np.allclose(nan_err.sigma_star_err, 0.0388 * nan_err.sigma_star, rtol=1e-6)
fit_table.groupby(['arm', 'sample'], sort=False).agg(fits=('target', 'size'), valid=('valid', 'sum'))

fits  valid
arm     sample             
cn_free 194      194    194
        gained   154    153
plain   194      194    194
        gained   154    154

- Fit quality and posterior medians, gained against the 194: 16, 50, 84 % and two-sample KS p.
- `chi2px_raw`: catalogue errors only. `chi2px_fixed`: with a 3 % floor for every fit.

In [10]:
QUALITY = ['chi2px_raw', 'chi2px_fixed', 'floor_pct_q50', 'phot_chi2_per_band', 'phot_pull_mean', 'phot_pull_max', 'ultravista_pull_mean',
           'broad_optical_pull_mean', 'irac_pull_mean', 'sigma_resid', 'lnz_err',
           'age_gyr_q50', 'afe_q50', 'feh_q50', 'cfe_q50', 'nfe_q50']
SAMPLES = {arm: (g[g['sample'] == '194'], g[g['sample'] == 'gained']) for arm, g in ok.groupby('arm', sort=False)}
rows, railing = [], {}
for arm, (ref, new_fits) in SAMPLES.items():
    for column in QUALITY:
        a, b = ref[column], new_fits[column]
        if a.notna().all():  # the plain arm has no [C/Fe], [N/Fe]
            rows.append(dict(arm=arm, quantity=column, **dict(zip(['q16_194', 'q50_194', 'q84_194'], np.quantile(a, [.16, .5, .84]))),
                             **dict(zip(['q16_gained', 'q50_gained', 'q84_gained'], np.quantile(b, [.16, .5, .84]))),
                             ks_p=ks_2samp(a, b).pvalue))
    for name, g in (('194', ref), ('gained', new_fits)):
        railing[arm, name] = dict(fits=len(g), **{c: int(g[c].sum()) for c in g.columns if c.endswith('_railed') and g[c].notna().all()},
                                  chi2px_fixed_above_3=int((g.chi2px_fixed > 3).sum()))
quality = pd.DataFrame(rows)
quality.to_csv(OUT / 'fit-quality.csv', index=False)
railing = pd.DataFrame(railing).T.rename_axis(['arm', 'sample'])
railing.to_csv(OUT / 'railing.csv')
display(quality.round(3), railing)

,arm,quantity,q16_194,q50_194,q84_194,q16_gained,q50_gained,q84_gained,ks_p
0,cn_free,chi2px_raw,1.420,1.885,2.485,1.566,1.899,2.393,0.062
1,cn_free,chi2px_fixed,0.908,1.245,1.758,0.899,1.277,1.640,0.751
2,cn_free,floor_pct_q50,2.341,3.766,7.032,2.421,3.909,6.193,0.462
3,cn_free,phot_chi2_per_band,0.454,1.070,1.978,0.460,1.210,1.986,0.832
4,cn_free,phot_pull_mean,-0.002,0.070,0.123,0.022,0.078,0.143,0.216
5,cn_free,phot_pull_max,1.586,2.509,3.441,1.618,2.586,3.383,0.887
6,cn_free,ultravista_pull_mean,0.004,0.472,1.061,0.007,0.582,1.093,0.592
7,cn_free,broad_optical_pull_mean,0.024,0.259,0.901,-0.017,0.424,0.957,0.605
8,cn_free,irac_pull_mean,-2.045,-0.803,0.083,-2.054,-1.216,0.123,0.638
9,cn_free,sigma_resid,-1.720,-0.959,-0.208,-1.624,-1.011,-0.142,0.908


fits  afe_railed  cfe_railed  log_f_calib_railed  feh_railed  \
arm     sample                                                                  
cn_free 194     194.0         3.0         4.0                19.0         3.0   
        gained  153.0         0.0         1.0                 7.0         0.0   
plain   194     194.0         5.0         NaN                20.0         3.0   
        gained  154.0         1.0         NaN                 8.0         0.0   

                nfe_railed  sigma_railed  chi2px_fixed_above_3  
arm     sample                                                  
cn_free 194            4.0           1.0                   7.0  
        gained         1.0           0.0                   1.0  
plain   194            NaN           0.0                   7.0  
        gained         NaN           0.0                   1.0

- Fitted photometry by COSMOS2015 class and catalogue: medians per galaxy, and the count with $\chi^2$ per band above 2.

In [11]:
PHOT = ['phot_chi2_per_band', 'phot_pull_mean', 'phot_pull_max', 'ultravista_pull_mean', 'broad_optical_pull_mean', 'irac_pull_mean']
by_class = ok.groupby(['arm', 'sample', 'cosmos2015', 'photometry'], sort=False)
photometry = by_class[PHOT].median().join(by_class.size().rename('fits')).join(
    by_class.phot_chi2_per_band.agg(lambda x: int((x > 2).sum())).rename('chi2_per_band_above_2')).reset_index()
photometry.to_csv(OUT / 'photometry-by-cosmos2015-class.csv', index=False)
photometry.round(3)

,arm,sample,cosmos2015,photometry,phot_chi2_per_band,phot_pull_mean,phot_pull_max,ultravista_pull_mean,broad_optical_pull_mean,irac_pull_mean,fits,chi2_per_band_above_2
0,cn_free,194,clean,cosmos2025,1.661,0.098,3.101,0.867,0.786,-1.732,99,25
1,cn_free,194,clean,cosmos2020_classic,0.508,0.041,1.762,0.203,0.073,-0.072,95,2
2,cn_free,gained,shallow only,cosmos2025,1.651,0.106,2.883,0.892,0.708,-1.791,74,16
3,cn_free,gained,clean,cosmos2025,1.773,0.175,2.763,0.754,1.184,-1.667,1,0
4,cn_free,gained,shallow only,cosmos2020_classic,0.497,0.054,1.724,0.081,0.051,-0.010,49,1
5,cn_free,gained,saturation or bad area,cosmos2025,1.959,0.135,3.557,0.967,0.714,-1.932,10,5
6,cn_free,gained,saturation or bad area,cosmos2020_classic,0.606,0.032,1.933,0.188,0.023,-0.060,4,1
7,cn_free,gained,clean,cosmos2020_classic,0.565,0.044,2.124,0.210,-0.007,0.018,15,0
8,plain,194,clean,cosmos2025,1.658,0.098,3.036,0.819,0.796,-1.681,99,28
9,plain,194,clean,cosmos2020_classic,0.507,0.040,1.769,0.145,0.090,-0.045,95,1


## Age–redshift slope and $H$
- Estimator of `results/carbon-mask-window-2026-11-13/analysis.ipynb`: least squares slope, 10000 bootstrap draws, $H = -977.79 / ((1+z)\,\mathrm{slope})$.
- $z_\mathrm{eff}$ from `effective_redshift` in `src/chronometer.py`: Planck18 $dt/dz$ equals the slope of a line fitted to Planck18 ages at the sample redshifts.
- $H$ at $z_\mathrm{eff}$; $H$ at the median $z$ kept for earlier tables.

In [12]:
COSMO = resolve_cosmology(None)
H_UNIT = (1 / u.Gyr).to_value(u.km / u.s / u.Mpc)


def hubble(slope, z):
    return -H_UNIT / ((1 + z) * slope) if slope < 0 else np.nan


def line_fit(z, y):
    fit = linregress(z, y)
    indices = np.random.default_rng(20261110).integers(0, len(z), size=(10000, len(z)))
    zc = z[indices] - z[indices].mean(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        draws = np.sum(zc * y[indices], axis=1) / np.sum(zc**2, axis=1)
    q16, q84 = np.nanquantile(draws, [0.16, 0.84])
    return dict(slope=fit.slope, q16=q16, q84=q84, scatter=np.std(y - fit.intercept - fit.slope * z, ddof=2), draws=draws)


def groups(ref, new_fits):
    both = pd.concat([ref, new_fits])
    return {'194': ref, 'gained': new_fits, 'combined': both,
            'gained: clean COSMOS2015': new_fits[new_fits.cosmos2015 == 'clean'],
            'gained: shallow only': new_fits[new_fits.cosmos2015 == 'shallow only'],
            'gained: saturation or bad area': new_fits[new_fits.cosmos2015 == 'saturation or bad area'],
            'combined without saturation or bad area': both[both.cosmos2015 != 'saturation or bad area'],
            'combined without any COSMOS2015 flag': both[both.cosmos2015 == 'clean']}


rows, fitted = [], {}
for arm, (ref, new_fits) in SAMPLES.items():
    for group, g in groups(ref, new_fits).items():
        z, age = g.z.to_numpy(), g.age_gyr_q50.to_numpy()
        z_med = float(np.median(z))
        s, universe = line_fit(z, age), line_fit(z, np.asarray(age_gyr(z, COSMO)))
        zeff = effective_redshift(z)
        z_eff = zeff['z_eff']
        fitted[arm, group] = s
        rows.append(dict(arm=arm, group=group, galaxies=len(g), z_min=z.min(), z_max=z.max(), slope_gyr_per_z=s['slope'],
                         bootstrap_q16=s['q16'], bootstrap_q84=s['q84'], residual_scatter_gyr=s['scatter'],
                         planck18_line_slope=universe['slope'],
                         slope_minus_planck_sigma=(s['slope'] - universe['slope']) / ((s['q84'] - s['q16']) / 2),
                         **zeff, H_zeff_kms_mpc=hubble(s['slope'], z_eff), H_zeff_from_q16=hubble(s['q16'], z_eff),
                         H_zeff_from_q84=hubble(s['q84'], z_eff), planck18_H_zeff=Planck18.H(z_eff).to_value(u.km / u.s / u.Mpc),
                         z_median=z_med, H_kms_mpc=hubble(s['slope'], z_med), H_from_q16=hubble(s['q16'], z_med),
                         H_from_q84=hubble(s['q84'], z_med),
                         median_age_gyr=g.age_gyr_q50.median(), median_afe=g.afe_q50.median(), median_feh=g.feh_q50.median()))
slopes = pd.DataFrame(rows)
slopes.to_csv(OUT / 'age-redshift-slopes.csv', index=False)
saved = pd.read_csv(ROOT / 'results/legac-dr3-2026-11-11/window-194-age-redshift-slopes.csv').query(
    "release == 'dr3' and group == 'all'").set_index('arm').slope_gyr_per_z
check = slopes[slopes.group == '194'].set_index('arm')
assert np.allclose(check.slope_gyr_per_z, saved[check.index])
assert (check.z_eff.round(3) == 0.775).all()
slopes.round(3)

,arm,group,galaxies,z_min,z_max,slope_gyr_per_z,bootstrap_q16,bootstrap_q84,residual_scatter_gyr,planck18_line_slope,...,H_zeff_from_q16,H_zeff_from_q84,planck18_H_zeff,z_median,H_kms_mpc,H_from_q16,H_from_q84,median_age_gyr,median_afe,median_feh
0,cn_free,194,194,0.603,0.999,-5.154,-5.782,-4.534,0.805,-5.226,...,95.290,121.519,105.434,0.728,109.785,97.851,124.786,4.879,0.160,-0.325
1,cn_free,gained,153,0.607,0.982,-4.761,-5.473,-4.029,0.759,-5.240,...,100.803,136.925,105.288,0.728,118.843,103.386,140.433,4.802,0.141,-0.336
2,cn_free,combined,347,0.603,0.999,-5.011,-5.481,-4.533,0.783,-5.231,...,100.562,121.595,105.383,0.728,112.922,103.225,124.814,4.844,0.151,-0.332
3,cn_free,gained: clean COSMOS2015,16,0.659,0.878,-1.038,-3.073,1.084,0.761,-5.302,...,180.539,NaN,104.642,0.704,552.968,186.756,NaN,4.493,0.194,-0.271
4,cn_free,gained: shallow only,123,0.607,0.982,-5.274,-6.046,-4.449,0.759,-5.254,...,91.362,124.169,105.142,0.727,107.357,93.652,127.281,4.937,0.138,-0.346
5,cn_free,gained: saturation or bad area,14,0.695,0.928,-4.729,-6.857,-2.603,0.659,-5.039,...,78.962,208.013,107.442,0.799,114.909,79.245,208.760,4.443,0.063,-0.355
6,cn_free,combined without saturation or bad area,333,0.603,0.999,-5.091,-5.583,-4.608,0.788,-5.237,...,98.789,119.683,105.319,0.728,111.161,101.372,122.812,4.845,0.153,-0.331
7,cn_free,combined without any COSMOS2015 flag,210,0.603,0.999,-4.932,-5.541,-4.355,0.806,-5.229,...,99.475,126.556,105.398,0.728,114.714,102.123,129.925,4.841,0.167,-0.323
8,plain,194,194,0.603,0.999,-5.043,-5.702,-4.388,0.875,-5.226,...,96.620,125.565,105.434,0.728,112.186,99.218,128.941,4.879,0.131,-0.317
9,plain,gained,154,0.607,0.982,-4.309,-5.001,-3.569,0.743,-5.241,...,110.337,154.602,105.276,0.728,131.317,113.165,158.564,4.824,0.139,-0.324


- Gained slope minus the 194 slope; the two samples share no galaxy, so the bootstrap widths add in quadrature.

In [13]:
rows = []
for arm in SAMPLES:
    for group in ('gained', 'gained: shallow only', 'gained: saturation or bad area', 'gained: clean COSMOS2015'):
        a, b = fitted[arm, group], fitted[arm, '194']
        width = np.hypot((a['q84'] - a['q16']) / 2, (b['q84'] - b['q16']) / 2)
        rows.append(dict(arm=arm, group=group, reference='194', slope_difference=a['slope'] - b['slope'], width=width,
                         difference_sigma=(a['slope'] - b['slope']) / width))
differences = pd.DataFrame(rows)
differences.to_csv(OUT / 'slope-differences.csv', index=False)
differences.round(3)

,arm,group,reference,slope_difference,width,difference_sigma
0,cn_free,gained,194,0.393,0.954,0.411
1,cn_free,gained: shallow only,194,-0.121,1.014,-0.119
2,cn_free,gained: saturation or bad area,194,0.425,2.217,0.192
3,cn_free,gained: clean COSMOS2015,194,4.116,2.170,1.896
4,plain,gained,194,0.734,0.972,0.755
5,plain,gained: shallow only,194,0.267,1.030,0.259
6,plain,gained: saturation or bad area,194,0.768,2.041,0.376
7,plain,gained: clean COSMOS2015,194,4.567,2.545,1.794
